# Room Booking Assistant - Technology and Evaluation Notebook

This notebook documents the technologies used in the Promtior technical challenge and shows how they are applied to the deployed solution.

- **Source:** [github.com/jimagrini/room-booking-assistant](https://github.com/jimagrini/room-booking-assistant)
- **Live application:** [room-booking-assistant-production.up.railway.app](https://room-booking-assistant-production.up.railway.app/)
- **Evaluation date:** 2026-09-03

> The notebook contains no API keys, JWT secrets, database passwords, or shared login passwords. Authentication is requested interactively when executable cells are run.

## 1. Challenge objective and implemented scope

The objective is to provide a conversational meeting-room booking system with tool-calling capabilities. The completed solution supports:

- Authentication for the two challenge users.
- Availability searches across rooms A-E.
- Room-specific capacity validation.
- Contiguous 30-minute slots and a maximum booking duration of three hours.
- Schedule retrieval for a specific room.
- Booking creation for the authenticated user.
- Listing and cancellation of bookings owned by that user.
- Protection against overlapping bookings.
- A responsive web interface and a public cloud deployment.

## 2. Solution architecture

| Layer | Responsibility | Technology |
| --- | --- | --- |
| Web | Login and conversational interface | React 19, TypeScript, Vite |
| API | HTTP endpoints, JWT validation, error handling, static SPA hosting | ASP.NET Core, .NET 10 |
| Assistant | Prompt orchestration, conversation isolation, tool loop | Groq Responses API, C# |
| Application | Booking use cases and request validation | C# application services |
| Domain | Booking invariants and entities | C# domain model |
| Infrastructure | Persistence, migrations, repositories, concurrency protection | EF Core, PostgreSQL |
| Runtime | Reproducible build and public hosting | Docker, Railway |

A key design decision is that the language model never writes directly to the database. It selects a strict tool, while deterministic application services enforce authorization and business rules.

## 3. Why these technologies were selected

- **.NET 10 and ASP.NET Core:** strong typing, dependency injection, authentication middleware, and production-ready HTTP APIs.
- **Entity Framework Core and PostgreSQL:** migrations, transactional persistence, and database-level uniqueness for booking slots.
- **JWT:** stateless authentication between the React client and API.
- **Groq Responses API:** an OpenAI-compatible, cloud-friendly model provider with tool calling.
- **React, TypeScript, and Vite:** a typed, responsive single-page interface with a fast development workflow.
- **xUnit:** focused domain, application, API, assistant, and persistence tests.
- **Docker and Railway:** one reproducible image containing both web and API, connected privately to managed PostgreSQL.

## 4. Deterministic booking rules

The application normalizes input to UTC, requires half-hour boundaries, rejects invalid ranges, and enforces the three-hour maximum before persistence.

```csharp
private static BookingTimeRange Create(
    DateTimeOffset startTime,
    DateTimeOffset endTime,
    TimeSpan maximumDuration,
    string errorPrefix)
{
    var startTimeUtc = startTime.ToUniversalTime();
    var endTimeUtc = endTime.ToUniversalTime();

    EnsureAligned(startTimeUtc, errorPrefix);
    EnsureAligned(endTimeUtc, errorPrefix);

    if (endTimeUtc <= startTimeUtc)
        throw new RequestValidationException(
            $"{errorPrefix}.invalid_time_range",
            "End time must be after start time.");

    var duration = endTimeUtc - startTimeUtc;
    if (duration > maximumDuration)
        throw new RequestValidationException(
            $"{errorPrefix}.duration_exceeded",
            "The requested time range exceeds the limit.");

    return new BookingTimeRange(startTimeUtc, endTimeUtc);
}
```

Source: [`BookingTimeRange.cs`](https://github.com/jimagrini/room-booking-assistant/blob/main/src/RoomBooking.Application/Bookings/BookingTimeRange.cs)

## 5. PostgreSQL persistence and overlap protection

Each reservation is expanded into 30-minute `BookingSlot` rows. A unique database index on `(RoomId, StartTimeUtc)` prevents two concurrent requests from claiming the same room slot, even if both pass an earlier availability check.

```csharp
builder.HasIndex(slot => new
    {
        slot.RoomId,
        slot.StartTimeUtc
    })
    .IsUnique()
    .HasDatabaseName("ux_booking_slots_room_start");
```

Source: [`BookingSlotConfiguration.cs`](https://github.com/jimagrini/room-booking-assistant/blob/main/src/RoomBooking.Infrastructure/Persistence/Configurations/BookingSlotConfiguration.cs)

Entity Framework migrations are applied at application startup, and rooms A-E are seeded idempotently.

## 6. Authentication and user isolation

The login endpoint verifies one of the challenge users and returns a signed JWT. Protected endpoints derive the user identity from the validated token; they never accept a user ID supplied by the browser or language model.

Conversation state is also associated with that authenticated user. Reusing another user's `conversationId` returns a not-found response instead of exposing conversation history. Booking cancellation is restricted to reservations owned by the caller.

## 7. Strict assistant tools

The assistant exposes five explicit functions:

| Tool | Purpose |
| --- | --- |
| `list_available_rooms` | Find rooms free for the full interval with enough capacity |
| `get_room_schedule` | Return available and occupied 30-minute slots |
| `list_my_bookings` | Retrieve bookings owned by the authenticated user |
| `create_booking` | Create a validated booking for the authenticated user |
| `cancel_booking` | Cancel an owned active booking using its UUID |

Tool schemas use `strict: true`, required fields, enums for room names, and `additionalProperties: false`. For example:

```json
{
  "type": "function",
  "name": "list_available_rooms",
  "strict": true,
  "parameters": {
    "type": "object",
    "required": ["start_time", "end_time", "attendee_count"],
    "additionalProperties": false
  }
}
```

Source: [`AssistantToolCatalog.cs`](https://github.com/jimagrini/room-booking-assistant/blob/main/src/RoomBooking.Api/Assistant/AssistantToolCatalog.cs)

## 8. Tool-calling orchestration

The service sends the user message and isolated conversation history to Groq. If the model requests functions, the server executes them and adds authoritative outputs to the next model request. A final natural-language answer is returned only after tool execution.

```csharp
if (response.FunctionCalls.Count == 0)
{
    return new AssistantMessageResponse(
        conversation.ConversationId,
        response.OutputText,
        toolsUsed.Distinct(StringComparer.Ordinal).ToArray());
}

foreach (var functionCall in response.FunctionCalls)
{
    var output = await toolExecutor.ExecuteAsync(
        functionCall, cancellationToken);
    inputItems.Add(JsonSerializer.SerializeToElement(new
    {
        type = "function_call_output",
        call_id = functionCall.CallId,
        output
    }));
    toolsUsed.Add(functionCall.Name);
}
```

The production implementation additionally validates empty responses, limits the loop to eight rounds, preserves provider output items, and reports stable API errors.

Source: [`AssistantService.cs`](https://github.com/jimagrini/room-booking-assistant/blob/main/src/RoomBooking.Api/Assistant/AssistantService.cs)

## 9. Web client and deployment

The React client stores the JWT only for the current browser session, sends authenticated assistant messages, preserves the returned `conversationId`, renders lightweight Markdown, and exposes loading and error states.

The root multi-stage Docker build:

1. Builds the React client with Node.js.
2. Publishes the .NET 10 API.
3. Copies the Vite output into ASP.NET Core `wwwroot`.
4. Starts the API on Railway's injected `PORT`.

This makes the browser and API share one HTTPS origin. Railway PostgreSQL remains on the project's private network, and only the application receives a public domain.

## 10. Executable production walkthrough

The following Python cells call the public deployment using only Python's standard library. They:

1. Check service health.
2. Request the challenge password interactively.
3. Authenticate and keep the JWT only in notebook memory.
4. Ask the assistant for room availability.
5. Optionally run a create-list-cancel-release lifecycle.

> Run the cells in order. Leave `RUN_MUTATING_DEMO = False` unless you intentionally want to create and then cancel a temporary production booking.

In [ ]:
import json
import os
import uuid
from datetime import date, timedelta
from getpass import getpass
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

BASE_URL = os.getenv(
    "ROOM_BOOKING_BASE_URL",
    "https://room-booking-assistant-production.up.railway.app",
).rstrip("/")

def api_request(method, path, payload=None, token=None, timeout=75):
    body = None
    headers = {"Accept": "application/json"}

    if payload is not None:
        body = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"

    if token:
        headers["Authorization"] = f"Bearer {token}"

    request = Request(
        f"{BASE_URL}{path}",
        data=body,
        headers=headers,
        method=method,
    )

    try:
        with urlopen(request, timeout=timeout) as response:
            content = response.read().decode("utf-8")
            return json.loads(content) if content else None
    except HTTPError as exception:
        content = exception.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"HTTP {exception.code} calling {path}: {content}"
        ) from exception
    except URLError as exception:
        raise RuntimeError(f"Could not reach {BASE_URL}: {exception}") from exception

print(f"Target API: {BASE_URL}")

### 10.1 Health check

A healthy response proves that the deployed ASP.NET Core process started after connecting to PostgreSQL and applying migrations.

In [ ]:
health = api_request("GET", "/health")
assert health == {"status": "healthy"}, health
health

### 10.2 Login

The password is read from `ROOM_BOOKING_PASSWORD` when available; otherwise `getpass` requests it without displaying or storing it in the notebook.

In [ ]:
username = os.getenv("ROOM_BOOKING_USERNAME", "User1")
password = os.getenv("ROOM_BOOKING_PASSWORD") or getpass(
    "Challenge password (input hidden): "
)

login = api_request(
    "POST",
    "/api/auth/login",
    {"username": username, "password": password},
)
access_token = login["accessToken"]

{
    "authenticatedUser": login["user"],
    "expiresAtUtc": login["expiresAtUtc"],
}

### 10.3 Conversation helper

The first message omits `conversationId`; later messages reuse the UUID returned by the API. The token and conversation identifier remain only in memory.

In [ ]:
conversation_id = None

def ask_assistant(message):
    global conversation_id

    payload = {"message": message}
    if conversation_id is not None:
        payload["conversationId"] = conversation_id

    response = api_request(
        "POST",
        "/api/assistant/messages",
        payload,
        token=access_token,
    )
    conversation_id = response["conversationId"]

    print(response["message"])
    print("\nTools used:", response["toolsUsed"])
    return response

In [ ]:
target_date = (date.today() + timedelta(days=7)).isoformat()

availability = ask_assistant(
    f"Which rooms are available on {target_date} "
    "from 13:00 to 14:00 for 5 people?"
)
assert "list_available_rooms" in availability["toolsUsed"]

### 10.4 Optional booking lifecycle

This cell is disabled by default because it changes production data. When enabled, it uses a unique title, creates a booking, lists it, cancels it, and verifies released availability through the assistant.

In [ ]:
RUN_MUTATING_DEMO = False
demo_title = f"Notebook demo {uuid.uuid4().hex[:8]}"

if RUN_MUTATING_DEMO:
    created = ask_assistant(
        f"Book the smallest available room on {target_date} "
        f"from 13:00 to 14:00 with title '{demo_title}' for 5 attendees."
    )
    assert "create_booking" in created["toolsUsed"]

    listed = ask_assistant("List my active bookings.")
    assert "list_my_bookings" in listed["toolsUsed"]

    cancelled = ask_assistant(f"Cancel my booking titled '{demo_title}'.")
    assert "cancel_booking" in cancelled["toolsUsed"]

    released = ask_assistant(
        f"Check room availability again on {target_date} "
        "from 13:00 to 14:00 for 5 people."
    )
    assert "list_available_rooms" in released["toolsUsed"]
else:
    print("Mutation demo skipped. Set RUN_MUTATING_DEMO = True to execute it.")

## 11. Test strategy and evidence

The repository contains 34 automated tests covering:

- Domain validation for rooms, capacity, duration, and slot alignment.
- Application use cases for availability, schedules, creation, ownership, and cancellation.
- API authentication behavior.
- Assistant orchestration, tool rounds, conversation isolation, and provider error handling.
- EF Core persistence and database conflict protection.

Validation completed on 2026-09-03:

| Check | Result |
| --- | --- |
| `dotnet build --no-restore` | Passed |
| `dotnet test --no-build` | 34 passed, 0 failed, 0 skipped |
| `npm run build` | Passed |
| `npm run lint` | Passed |
| `docker build -t room-booking-assistant .` | Passed |
| Production smoke test | Health, login, availability, create, list, cancel, and release passed |

## 12. Design decisions and trade-offs

- **Deterministic rules over model judgment:** prevents the LLM from bypassing booking constraints.
- **30-minute slot rows:** simplifies availability queries and gives PostgreSQL a strong concurrency invariant.
- **UTC persistence with Montevideo presentation:** avoids ambiguous storage while keeping user-facing times local.
- **In-memory conversation history:** appropriate for the challenge and keeps provider integration stateless, but requires one application replica. A distributed cache would be the next scaling step.
- **Single web/API container:** simplifies deployment and removes production CORS complexity.
- **Secrets only at runtime:** Groq, JWT, database, and login secrets are Railway variables or local environment variables, never repository content.

## 13. Conclusion

The solution meets the functional requirements of the challenge and is publicly deployed. The assistant uses the language model for intent interpretation while authentication, authorization, persistence, capacity, time alignment, maximum duration, and overlap prevention remain deterministic and testable.

The live walkthrough above provides a reproducible way to verify the deployed integration without embedding secrets or modifying production data by default.